In [4]:
import pandas as pd
import random

# -------------------
# 1. Load CSV mapping
# -------------------
df = pd.read_csv("generic.csv")

In [6]:
df.head()

,dtc,description
0,DTC Codes - P0100-P0199 – Fuel and Air Metering,NaN
1,P0100,Mass or Volume Air Flow Circuit Malfunction
2,P0101,Mass or Volume Air Flow Circuit Range/Performa...
3,P0102,Mass or Volume Air Flow Circuit Low Input
4,P0103,Mass or Volume Air Flow Circuit High Input


In [8]:
print(df.columns)

Index(['dtc', ' description'], dtype='object')


In [10]:
df.columns = df.columns.str.strip()


In [12]:
print(df.columns)

Index(['dtc', 'description'], dtype='object')


In [14]:


# Clean mapping: only keep rows where dtc is a real code (skip headers/NaN)
fault_type_map = df.dropna(subset=["description"])
fault_type_map = fault_type_map[~fault_type_map["dtc"].str.contains("DTC", case=False, na=False)]
fault_type_dict = dict(zip(fault_type_map["dtc"].str.strip(), fault_type_map["description"].str.strip()))

# -------------------
# 2. Supporting lists
# -------------------
equipment_map = {
    "P": ["Engine", "Turbocharger System", "Fuel System"],
    "C": ["Brake System", "Suspension", "Steering System"],
    "B": ["Airbag System", "Climate Control", "Door Locks"],
    "U": ["Communication Network", "Control Module Network"],
}

maintenance_methods = ["Replace", "Inspect", "Clean", "Calibrate", "Tighten"]
locations = [
    "Engine Bay", "Front Left Wheel", "Front Right Wheel",
    "Rear Left Wheel", "Rear Right Wheel", "Dashboard"
]
device_properties_samples = [
    "Diesel, 2.0L, 4-cylinder",
    "Petrol, 1.6L, turbocharged",
    "Hybrid, 2.5L",
    "Electric, dual motor",
    "Petrol, 3.0L, V6"
]
symptoms_samples = [
    "Loss of power",
    "ABS warning light on",
    "Airbag light illuminated",
    "Engine misfire",
    "Check engine light on",
    "Rough idling",
    "Unresponsive throttle"
]
brands = ["To", "Fo", "Ho", "Ni", "Hy", "Ki", "Bm", "Au", "Mz"]

def generate_vehicle_make_model():
    return f"variants:4KR0{random.randint(1,9)}{random.randint(1000,9999)}{random.randint(1000000,9999999)}{random.choice(brands)}"

# -------------------
# 3. Generator
# -------------------
def generate_synthetic_fault_data(n_records=1000):
    synthetic_records = []
    all_fault_codes = list(fault_type_dict.keys())

    for _ in range(n_records):
        fault_code = random.choice(all_fault_codes)
        fault_type = fault_type_dict.get(fault_code, "Unknown Fault Type")
        prefix = fault_code[0] if fault_code else "P"  # default to P if missing

        equipment_name = random.choice(equipment_map.get(prefix, ["General Vehicle System"]))
        maintenance_method = random.choice(maintenance_methods)
        location = random.choice(locations)
        device_properties = random.choice(device_properties_samples)

        # Measurement value logic
        if "Turbo" in fault_type or "Boost" in fault_type:
            measurement_value = f"{round(random.uniform(0.5, 3.0), 1)} bar"
        elif "Temperature" in fault_type or "Overheat" in fault_type:
            measurement_value = f"{random.randint(80, 120)}°C"
        elif "Voltage" in fault_type:
            measurement_value = f"{round(random.uniform(11.0, 14.5), 1)} V"
        else:
            measurement_value = f"{random.randint(0, 5000)} units"

        vehicle_make_model = generate_vehicle_make_model()
        time_duration = f"{random.randint(1, 5)} hours"
        symptom = random.choice(symptoms_samples)

        synthetic_records.append({
            "EquipmentName": equipment_name,
            "DeviceProperties": device_properties,
            "FaultCode": fault_code,
            "FaultType": fault_type,
            "MaintenanceMethod": maintenance_method,
            "VehicleComponentLocation": location,
            "MeasurementValue": measurement_value,
            "VehicleMakeModel": vehicle_make_model,
            "TimeDuration": time_duration,
            "CustomerReportedSymptom": symptom
        })

    return pd.DataFrame(synthetic_records)

# -------------------
# 4. Generate and view
# -------------------
synthetic_df = generate_synthetic_fault_data(10000)

# Save to CSV if needed
synthetic_df.to_csv("synthetic_fault_data_1.csv", index=False)

synthetic_df.head()


,EquipmentName,DeviceProperties,FaultCode,FaultType,MaintenanceMethod,VehicleComponentLocation,MeasurementValue,VehicleMakeModel,TimeDuration,CustomerReportedSymptom
0,Engine,"Petrol, 3.0L, V6",P0256,"Injection Pump Fuel Metering Control ""B"" Malfu...",Replace,Engine Bay,4317 units,variants:4KR0496669655996Ki,5 hours,Rough idling
1,Engine,"Diesel, 2.0L, 4-cylinder",P0770,Shift Solenoid E Malfunction,Calibrate,Rear Left Wheel,860 units,variants:4KR0233505000287Ni,5 hours,Airbag light illuminated
2,Engine,"Hybrid, 2.5L",P0342,Camshaft Position Sensor Circuit Low Input,Clean,Dashboard,2340 units,variants:4KR0597474545653To,2 hours,ABS warning light on
3,Fuel System,"Petrol, 3.0L, V6",P0603,Internal Control Module Keep Alive Memory (KAM...,Calibrate,Engine Bay,1160 units,variants:4KR0250138993692Fo,2 hours,Check engine light on
4,Turbocharger System,"Electric, dual motor",P0567,Cruise Control Resume Signal Malfunction,Tighten,Dashboard,3359 units,variants:4KR0255826277440Ho,3 hours,Airbag light illuminated


In [16]:
synthetic_df.columns

Index(['EquipmentName', 'DeviceProperties', 'FaultCode', 'FaultType',
       'MaintenanceMethod', 'VehicleComponentLocation', 'MeasurementValue',
       'VehicleMakeModel', 'TimeDuration', 'CustomerReportedSymptom'],
      dtype='object')

In [18]:
import pandas as pd
import random

# Assume synthetic_df is already loaded

# Some alternative phrases to make sentences more natural
fault_phrases = ["reported fault code", "detected fault", "identified issue with code"]
measurement_phrases = ["Measurement is", "Observed value:", "Reading:"]
duration_phrases = ["Duration recorded:", "Time taken:", "Elapsed time:"]
symptom_phrases = ["Customer observed:", "Symptom reported:", "User experienced:"]

def generate_realistic_ner_record(row):
    # Shuffle clauses
    clauses = [
        f"The {row['VehicleMakeModel']} has an {row['EquipmentName']} with {row['DeviceProperties']}.",
        f"{random.choice(fault_phrases)} {row['FaultCode']} ({row['FaultType']}) at {row['VehicleComponentLocation']}.",
        f"Maintenance method: {row['MaintenanceMethod']}.",
        f"{random.choice(measurement_phrases)} {row['MeasurementValue']}.",
        f"{random.choice(duration_phrases)} {row['TimeDuration']}.",
        f"{random.choice(symptom_phrases)} {row['CustomerReportedSymptom']}."
    ]
    random.shuffle(clauses)  # shuffle to create variety
    text = " ".join(clauses)

    spans = []
    # Use column names directly as labels
    for col in synthetic_df.columns:
        value = str(row[col])
        start = text.find(value)
        if start != -1:
            end = start + len(value)
            spans.append([start, end, col])  # label = column name

    return {"text": text, "entities": spans}

# Generate NER dataset
ner_data_realistic = [generate_realistic_ner_record(row) for _, row in synthetic_df.iterrows()]

# Convert to DataFrame or save directly
ner_df_realistic = pd.DataFrame(ner_data_realistic)
#ner_df_realistic.to_json("synthetic_ner_dataset_realistic.json", orient="records", lines=True)

#print(f"Realistic synthetic NER dataset generated with {len(ner_df_realistic)} records.")


In [20]:
ner_df_realistic.head()

,text,entities
0,Observed value: 4317 units. The variants:4KR04...,"[[67, 73, EquipmentName], [79, 95, DevicePrope..."
1,Duration recorded: 5 hours. identified issue w...,"[[196, 202, EquipmentName], [208, 232, DeviceP..."
2,identified issue with code P0342 (Camshaft Pos...,"[[199, 205, EquipmentName], [211, 223, DeviceP..."
3,Observed value: 1160 units. Maintenance method...,"[[225, 236, EquipmentName], [242, 258, DeviceP..."
4,The variants:4KR0255826277440Ho has an Turboch...,"[[39, 58, EquipmentName], [64, 84, DevicePrope..."


In [22]:
ner_df_realistic['text'][0]

'Observed value: 4317 units. The variants:4KR0496669655996Ki has an Engine with Petrol, 3.0L, V6. Symptom reported: Rough idling. reported fault code P0256 (Injection Pump Fuel Metering Control "B" Malfunction (Cam/Rotor/Injector)) at Engine Bay. Time taken: 5 hours. Maintenance method: Replace.'

In [34]:
ner_df_realistic['entities'][0]

[[67, 73, 'EquipmentName'],
 [79, 95, 'DeviceProperties'],
 [149, 154, 'FaultCode'],
 [156, 229, 'FaultType'],
 [287, 294, 'MaintenanceMethod'],
 [234, 244, 'VehicleComponentLocation'],
 [16, 26, 'MeasurementValue'],
 [32, 59, 'VehicleMakeModel'],
 [258, 265, 'TimeDuration'],
 [115, 127, 'CustomerReportedSymptom']]

In [26]:
ner_df_realistic.to_json("synthetic_ner_dataset_realistic.json", orient="records", lines=True)

#print(f"Realistic synthetic NER dataset generated with {len(ner_df_realistic)} records.")

In [36]:
import json
import pandas as pd

# Load your synthetic dataframe
# Example: synthetic_df = pd.read_csv("synthetic_df.csv")
# Assuming your dataframe has columns: 'text' and 'entities' (list of [start, end, label])
# If entities are in separate columns, you may need to construct them first

# For demonstration, let's assume 'synthetic_df' is already loaded
# and has 'text' and 'entities' columns
synthetic_df = ner_df_realistic
label_studio_data = []

for i, row in synthetic_df.iterrows():
    ls_record = {
        "id": i,
        "data": {
            "text": row["text"]
        },
        "annotations": [
            {
                "id": 0,
                "result": [
                    {
                        "from_name": "label",
                        "to_name": "text",
                        "type": "labels",
                        "value": {
                            "start": ent[0],
                            "end": ent[1],
                            "text": row["text"][ent[0]:ent[1]],
                            "labels": [ent[2]]
                        }
                    } for ent in row["entities"]
                ]
            }
        ]
    }
    label_studio_data.append(ls_record)

# Save the full dataset as a single JSON array
with open("14august_synthetic_ner_labelstudio.json", "w") as f:
    json.dump(label_studio_data, f, indent=2)

print(f"Converted {len(label_studio_data)} records to Label Studio JSON format.")


Converted 10000 records to Label Studio JSON format.
